<a href="https://colab.research.google.com/github/justii543/MLpreps/blob/main/CodeVulnerability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Dependencies

In [ ]:
!pip install transformers torch datasets pandas numpy scikit-learn

In [ ]:
!git clone https://github.com/DLVulDet/PrimeVul

In [ ]:
import os

# Check folder structure
os.listdir("PrimeVul")

Load CodeBERT

In [ ]:
import torch
import json
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "microsoft/codebert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

model.to(device)   # MOVE MODEL TO GPU
model.eval()

Embedding Function

In [ ]:
def get_function_embedding(code_snippet):
    inputs = tokenizer(
        code_snippet,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    cls_embedding = outputs.last_hidden_state[:, 0, :]

    return cls_embedding.squeeze().cpu().numpy()

Updated :Load Dataset - Both train_set and test_set

In [ ]:
def load_dataset(file_path):
    data = []
    with open(file_path, "r") as f:
        for line in f:
            data.append(json.loads(line))
    return data

train_path = "PrimeVul/primevul_train.jsonl"
test_path = "PrimeVul/primevul_test.jsonl"

train_dataset = load_dataset(train_path)
test_dataset = load_dataset(test_path)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

Collect Balanced Train dataset

In [ ]:
balanced_train = []

count_0 = 0
count_1 = 0
limit_per_class = 100   # you can increase later

for item in train_dataset:
    label = item["target"]

    if label == 0 and count_0 < limit_per_class:
        balanced_train.append(item)
        count_0 += 1

    elif label == 1 and count_1 < limit_per_class:
        balanced_train.append(item)
        count_1 += 1

    # Stop when both classes collected
    if count_0 >= limit_per_class and count_1 >= limit_per_class:
        break

print("Collected:", len(balanced_train))
print("Class 0:", count_0, "Class 1:", count_1)

Updated: Collect Balanced Test set

In [ ]:
balanced_test = []

count_0 = 0
count_1 = 0

limit_per_class = 50

for item in test_dataset:

    label = item["target"]

    if label == 0 and count_0 < limit_per_class:
        balanced_test.append(item)
        count_0 += 1

    elif label == 1 and count_1 < limit_per_class:
        balanced_test.append(item)
        count_1 += 1

    if count_0 >= limit_per_class and count_1 >= limit_per_class:
        break

print("Balanced Test Dataset:", len(balanced_test))
print("Class 0:", count_0)
print("Class 1:", count_1)


Generate Train Embeddings (M1 Output)

In [ ]:
train_embeddings = []
train_labels = []

for item in tqdm(balanced_train):

    code = item["func"]
    label = item["target"]

    emb = get_function_embedding(code)

    train_embeddings.append(emb)
    train_labels.append(label)

train_embeddings = np.array(train_embeddings)
train_labels = np.array(train_labels)

# FIX SHAPE
train_embeddings = train_embeddings.reshape(train_embeddings.shape[0], -1)

print("Train Embeddings Shape:", train_embeddings.shape)

Generate Test Embeddings

In [ ]:
test_embeddings = []
test_labels = []

for item in tqdm(balanced_test):

    code = item["func"]
    label = item["target"]

    emb = get_function_embedding(code)

    test_embeddings.append(emb)
    test_labels.append(label)

test_embeddings = np.array(test_embeddings)
test_labels = np.array(test_labels)

# FIX SHAPE
test_embeddings = test_embeddings.reshape(test_embeddings.shape[0], -1)

print("Test Embeddings Shape:", test_embeddings.shape)

Train Model

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(kernel='linear', class_weight='balanced')

svm_model.fit(train_embeddings, train_labels)

Evaluate Model

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

predictions = svm_model.predict(test_embeddings)

accuracy = accuracy_score(test_labels, predictions)
print("Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(test_labels, predictions))

In [ ]:
code_example = """
int vulnerable(char *input) {
    char buffer[10];
    strcpy(buffer, input);
    return 0;
}
"""

emb = get_function_embedding(code_example)
prediction = svm_model.predict([emb])

print("Prediction:", prediction[0])
# 1 = vulnerable, 0 = safe